In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Build Stacking Ensemble (Meta‑Learner)

In [ ]:
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load features and splits
X_train_final, X_test_final = joblib.load('/kaggle/working/feature_matrices.pkl')
_, _, y_train, y_test = joblib.load('/kaggle/working/data_splits.pkl')

# Load base models
lr = joblib.load('/kaggle/working/model_lr.pkl')
xgb = joblib.load('/kaggle/working/model_xgb.pkl')
lgbm = joblib.load('/kaggle/working/model_lgbm.pkl')

print("Base models loaded.")

In [ ]:
# Generate probability predictions for meta‑features

# Get predicted probabilities for class 1 (True news)
p_lr_train = lr.predict_proba(X_train_final)[:, 1].reshape(-1, 1)
p_lr_test = lr.predict_proba(X_test_final)[:, 1].reshape(-1, 1)

p_xgb_train = xgb.predict_proba(X_train_final)[:, 1].reshape(-1, 1)
p_xgb_test = xgb.predict_proba(X_test_final)[:, 1].reshape(-1, 1)

p_lgbm_train = lgbm.predict_proba(X_train_final)[:, 1].reshape(-1, 1)
p_lgbm_test = lgbm.predict_proba(X_test_final)[:, 1].reshape(-1, 1)

# Stack as features for meta‑learner
meta_train = np.hstack([p_lr_train, p_xgb_train, p_lgbm_train])
meta_test = np.hstack([p_lr_test, p_xgb_test, p_lgbm_test])

print(f"Meta-train shape: {meta_train.shape}")
print(f"Meta-test shape: {meta_test.shape}")

**Train meta‑learner (Logistic Regression) and evaluate**

In [ ]:
# Train meta‑learner (Logistic Regression) and evaluate

# Meta-learner
meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_model.fit(meta_train, y_train)

# Final predictions
y_pred_stack = meta_model.predict(meta_test)
acc_stack = accuracy_score(y_test, y_pred_stack)

print(f"\n🎯 Stacking Ensemble Accuracy: {acc_stack:.4f} ({acc_stack*100:.4f}%)")

mis = (y_pred_stack != y_test).sum()
print(f"Misclassified: {mis} out of {len(y_test)}")

# Save meta-model
joblib.dump(meta_model, '/kaggle/working/meta_model.pkl')
print("Saved: meta_model.pkl")

**ensemble evaluation**

In [ ]:
# Ensure acc_stack is defined (if not, compute it first)
# Example:
# y_pred_stack = meta_model.predict(meta_test)
# acc_stack = accuracy_score(y_test, y_pred_stack)

print("\n" + "="*50)
print(f"{'MODEL':<30} {'ACCURACY (%)':>15}")
print("="*50)
print(f"{'Logistic Regression':<30} {99.5073:>15.4f}")
print(f"{'XGBoost':<30} {99.6370:>15.4f}")
print(f"{'LightGBM':<30} {99.6370:>15.4f}")
print(f"{'Stacking Ensemble':<30} {acc_stack*100:>15.4f}")
print(f"{'Baseline Deep Learning Framework':<30} {99.13:>15.2f}")
print("="*50)

**comparison plots**

In [ ]:
import matplotlib.pyplot as plt

# Model names
models = [
    'Logistic\nRegression',
    'XGBoost',
    'LightGBM',
    'Stacking\nEnsemble',
    'Baseline\nFramework'
]

# Accuracy values
accuracies = [
    99.5073,
    99.6370,
    99.6370,
    acc_stack * 100,
    99.13
]

# Create figure
plt.figure(figsize=(10, 6))

# Bar plot
bars = plt.bar(models, accuracies)

# Add value labels on top
for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.01,
        f'{acc:.2f}%',
        ha='center',
        fontsize=10
    )

# Labels and title
plt.ylabel('Accuracy (%)', fontsize=12)
plt.xlabel('Models', fontsize=12)
plt.title('Model Performance Comparison', fontsize=14)

# Y-axis limit
plt.ylim(98.8, 100)

# Grid
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Save figure
plt.savefig("model_comparison.png", dpi=300, bbox_inches='tight')

# Show plot
plt.show()

# ROC-AUC CURVE

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import re
from scipy.sparse import hstack, csr_matrix

# Load feature matrices and splits
X_train_final, X_test_final = joblib.load('/kaggle/working/feature_matrices.pkl')
_, _, y_train, y_test = joblib.load('/kaggle/working/data_splits.pkl')

# Load base models and meta-model
lr = joblib.load('/kaggle/working/model_lr.pkl')
xgb = joblib.load('/kaggle/working/model_xgb.pkl')
lgbm = joblib.load('/kaggle/working/model_lgbm.pkl')
meta_model = joblib.load('/kaggle/working/meta_model.pkl')

# Load vectorizers and scaler (for cross-dataset test later)
word_vec = joblib.load('/kaggle/working/word_vectorizer.pkl')
char_vec = joblib.load('/kaggle/working/char_vectorizer.pkl')
scaler = joblib.load('/kaggle/working/meta_scaler.pkl')

print("All models and data loaded successfully.")
print(f"Test set size: {len(y_test)}")

In [ ]:
# Get probabilities from base models for test set
p_lr = lr.predict_proba(X_test_final)[:, 1].reshape(-1, 1)
p_xgb = xgb.predict_proba(X_test_final)[:, 1].reshape(-1, 1)
p_lgbm = lgbm.predict_proba(X_test_final)[:, 1].reshape(-1, 1)

# Stack and get final probabilities from meta-model
meta_input = np.hstack([p_lr, p_xgb, p_lgbm])
y_proba = meta_model.predict_proba(meta_input)[:, 1]

# Calculate ROC-AUC
auc = roc_auc_score(y_test, y_proba)
print(f"Stacking Ensemble ROC-AUC: {auc:.4f}")

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Stacking Ensemble (AUC = {auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Stacking Ensemble')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('/kaggle/working/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roc_curve.png")

# Cross‑Dataset Test on LIAR Dataset

In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import re
import os
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

print("="*60)
print("DOWNLOADING LIAR DATASET FROM UCSB")
print("="*60)

# Official URL
url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"

try:
    # Download
    print("\nDownloading liar_dataset.zip...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    
    # Extract ZIP in memory
    print("Extracting files...")
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        print("Files in zip:", z.namelist())
        
        # Read TSV files
        train_df = pd.read_csv(z.open('train.tsv'), sep='\t', header=None)
        test_df = pd.read_csv(z.open('test.tsv'), sep='\t', header=None)
        valid_df = pd.read_csv(z.open('valid.tsv'), sep='\t', header=None)
    
    # Column names for LIAR dataset
    column_names = [
        'id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party',
        'barely_true_counts', 'false_counts', 'half_true_counts',
        'mostly_true_counts', 'pants_on_fire_counts', 'context'
    ]
    
    # Assign column names
    for df in [train_df, test_df, valid_df]:
        df.columns = column_names[:len(df.columns)]
    
    # Combine all splits
    df_liar = pd.concat([train_df, test_df, valid_df], axis=0).reset_index(drop=True)
    
    print(f"\n✅ LIAR loaded: {len(df_liar)} statements")
    print(f"Unique label values: {df_liar['label'].unique()}")
    
    # CORRECTED: Map string labels to binary (0=Fake, 1=True)
    # LIAR labels are strings: 'true', 'mostly-true', 'half-true', 'barely-true', 'false', 'pants-fire'
    def to_binary(label):
        if label in ['pants-fire', 'false', 'barely-true']:
            return 0  # Fake
        else:  # 'half-true', 'mostly-true', 'true'
            return 1  # True
    
    df_liar['label_binary'] = df_liar['label'].apply(to_binary)
    y_liar = df_liar['label_binary'].values
    X_liar_raw = df_liar['statement']
    
    print(f"Label distribution: Fake={(y_liar==0).sum()}, True={(y_liar==1).sum()}")
    
    # Clean text (same as training)
    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    X_liar_clean = X_liar_raw.apply(clean_text)
    
    # Extract features using saved vectorizers
    X_liar_word = word_vec.transform(X_liar_clean)
    X_liar_char = char_vec.transform(X_liar_clean)
    
    # Meta-features
    def get_meta(texts):
        features = []
        for t in texts:
            t = str(t)
            words = t.split()
            features.append([
                len(t),
                len(words),
                sum(1 for c in t if c.isupper()) / (len(t) + 1),
                t.count('!'),
                t.count('?'),
                sum(1 for c in t if c in '.,;:'),
                np.mean([len(w) for w in words]) if words else 0
            ])
        return np.array(features)
    
    meta_raw = get_meta(X_liar_clean)
    meta_scaled = scaler.transform(meta_raw)
    X_liar_meta = csr_matrix(meta_scaled)
    
    # Combine all features
    X_liar_combined = hstack([X_liar_word, X_liar_char, X_liar_meta])
    print(f"Feature shape: {X_liar_combined.shape}")
    
    # Get predictions from base models
    p_lr = lr.predict_proba(X_liar_combined)[:, 1].reshape(-1, 1)
    p_xgb = xgb.predict_proba(X_liar_combined)[:, 1].reshape(-1, 1)
    p_lgbm = lgbm.predict_proba(X_liar_combined)[:, 1].reshape(-1, 1)
    
    # Stack and predict using meta-model
    meta_input = np.hstack([p_lr, p_xgb, p_lgbm])
    y_pred = meta_model.predict(meta_input)
    y_proba = meta_model.predict_proba(meta_input)[:, 1]
    
    # Calculate results
    acc = accuracy_score(y_liar, y_pred)
    auc = roc_auc_score(y_liar, y_proba)
    
    print("\n" + "="*50)
    print("CROSS-DATASET RESULTS (LIAR)")
    print("="*50)
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"ROC-AUC: {auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_liar, y_pred, target_names=['Fake', 'True']))
    
    cm = confusion_matrix(y_liar, y_pred)
    print("\nConfusion Matrix:")
    print("              Predicted")
    print("              Fake   True")
    print(f"Actual Fake   {cm[0,0]:5d}  {cm[0,1]:5d}")
    print(f"       True    {cm[1,0]:5d}  {cm[1,1]:5d}")

except Exception as e:
    print(f"Error: {e}")
    print("\nSkipping LIAR test. Run robustness test instead.")